# encoder-decoder-symmetric — ex2: diagnose a broken encoder-decoder config and report the asymmetry

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `encoder-decoder-symmetric`. Running the final beacon cell reports progress against the `CNN: Encoder-decoder symmetric layout` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Encoder-decoder symmetric layout` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`encoder-decoder-symmetric`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "encoder-decoder-symmetric"
DD_SUBTOPIC = "CNN: Encoder-decoder symmetric layout"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Encoder-decoder symmetry — what BREAKS it

Ex1 BUILT a symmetric autoencoder where each encoder pool was mirrored by a decoder upsample. The deepening move is to ANALYZE a given config and report whether end-to-end shape is preserved — and if not, where the asymmetry lives.

**Per-stage shape arithmetic.** For each encoder stage with stride `s`, the spatial size divides by `s` (integer division). For each decoder stage with upsample factor `u`, the spatial size multiplies by `u`. End-to-end shape is preserved iff the product of encoder strides == product of decoder upsamples AND the input H/W is divisible by the encoder stride-product.

```python
enc_div = 1
for s in encoder_strides:
    enc_div *= s
dec_mul = 1
for u in decoder_upsamples:
    dec_mul *= u
# Symmetric iff enc_div == dec_mul AND H % enc_div == 0.
```

**Why divisibility matters even when products match.** A stride-2 conv on a 7×7 input rounds down to 3×3. Upsampling 3×3 by 2 gives 6×6 — NOT the original 7×7. The product matches but the intermediate truncation destroys the round-trip. This is the silent bug: shapes look 'symmetric' but the autoencoder output is off-by-one.

### Exercise 2 — diagnose a broken encoder-decoder config and report the asymmetry

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze an (encoder_strides, decoder_upsamples, input_size) config and return a diagnostic dict reporting whether end-to-end shape is preserved — and if not, whether the failure is a product mismatch, a divisibility violation, or both.
> Keywords: encoder-decoder, symmetry, shape-arithmetic, diagnostic
> ```

**KCs targeted:** `stride-product-must-equal-upsample-product`, `input-divisible-by-stride-product`

Implement `ex2_diagnose_symmetry(encoder_strides, decoder_upsamples, input_hw)`. The deepening variant of ex1.

Inputs:
- `encoder_strides`: `list[int]`, e.g. `[2, 2]` (two pool stages).
- `decoder_upsamples`: `list[int]`, e.g. `[2, 2]`.
- `input_hw`: `int`, the H = W of a square input.

Return a dict with EXACTLY these keys:

- `'enc_div'`: `int`, product of `encoder_strides` (1 if list is empty).
- `'dec_mul'`: `int`, product of `decoder_upsamples` (1 if list is empty).
- `'product_matches'`: `bool`, `enc_div == dec_mul`.
- `'input_divisible'`: `bool`, `input_hw % enc_div == 0`.
- `'predicted_output_hw'`: `int`, the spatial size you'd get if you ran this config: `(input_hw // enc_div) * dec_mul` — floor-division on encoder, multiplication on decoder.
- `'shape_preserved'`: `bool`, `predicted_output_hw == input_hw`. True iff BOTH `product_matches` and `input_divisible`.
- `'failure_reason'`: `str | None`. `None` if `shape_preserved` is True. Otherwise one of `'product_mismatch'`, `'not_divisible'`, or `'both'` (when product matches but divisibility fails AND there's a product mismatch, return `'both'`).

Do not raise on empty lists — treat empty as product 1 (identity).

In [ ]:
def ex2_diagnose_symmetry(encoder_strides, decoder_upsamples, input_hw):
    enc_div = 1
    for s in encoder_strides:
        enc_div *= s
    dec_mul = 1
    for u in decoder_upsamples:
        dec_mul *= u
    product_matches = (enc_div == dec_mul)
    input_divisible = (input_hw % enc_div == 0)
    predicted = (input_hw // enc_div) * dec_mul
    shape_preserved = product_matches and input_divisible
    if shape_preserved:
        failure_reason = None
    elif not product_matches and not input_divisible:
        failure_reason = 'both'
    elif not product_matches:
        failure_reason = 'product_mismatch'
    else:
        failure_reason = 'not_divisible'
    return {
        'enc_div': enc_div,
        'dec_mul': dec_mul,
        'product_matches': product_matches,
        'input_divisible': input_divisible,
        'predicted_output_hw': predicted,
        'shape_preserved': shape_preserved,
        'failure_reason': failure_reason,
    }


<details><summary>Solution</summary>

```python
def ex2_diagnose_symmetry(encoder_strides, decoder_upsamples, input_hw):
    enc_div = 1
    for s in encoder_strides:
        enc_div *= s
    dec_mul = 1
    for u in decoder_upsamples:
        dec_mul *= u
    product_matches = (enc_div == dec_mul)
    input_divisible = (input_hw % enc_div == 0)
    predicted = (input_hw // enc_div) * dec_mul
    shape_preserved = product_matches and input_divisible
    if shape_preserved:
        failure_reason = None
    elif not product_matches and not input_divisible:
        failure_reason = 'both'
    elif not product_matches:
        failure_reason = 'product_mismatch'
    else:
        failure_reason = 'not_divisible'
    return {
        'enc_div': enc_div,
        'dec_mul': dec_mul,
        'product_matches': product_matches,
        'input_divisible': input_divisible,
        'predicted_output_hw': predicted,
        'shape_preserved': shape_preserved,
        'failure_reason': failure_reason,
    }
```

**Why predicted uses `//` not `/`.** Conv with stride truncates — a 7×7 input with stride-2 gives 3×3, not 3.5×3.5. The `predicted_output_hw` formula has to mirror that truncation; otherwise the diagnostic disagrees with real model behavior.

**`'both'` is a distinct category.** A config with BOTH a product mismatch AND a non-divisible input fails in two ways. Reporting just one would let the user fix it (e.g. balance the upsamples) and then hit the second error in the next iteration. `'both'` is the 'fix both before re-running' signal.

**Empty list → product 1.** The neutral element of multiplication. No encoder/decoder stages == identity transformation. The check still works because `input_hw % 1 == 0` always.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()